In [1]:
import pandas as pd

# 1. MOTHERSHIP INBOUND EMAILS (12 Rows with deliberate tactical signatures)
emails_data = {
    'email_id': ['e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'e10', 'e11', 'e12'],
    'sender_address': [
        'hr@company-benefits.com',       # Signal: Brand new domain, targeting HR Director
        'security@paypal-verify.net',    # Signal: Typosquatting/Phishing, targeting Finance
        'boss@company.com',              # Signal: Legitimate internal email
        'newsletter@techcrunch.com',     # Signal: Legitimate external newsletter
        'admin@micros0ft.com',           # Signal: Lookalike domain (0 instead of o)
        'billing@aws-invoice.io',        # Signal: High-velocity spray attack part 1
        'accounting@aws-invoice.io',     # Signal: High-velocity spray attack part 2 (same domain, 1 min later)
        'support@aws-invoice.io',        # Signal: High-velocity spray attack part 3 (same domain, same minute)
        'boss@company-urgent.com',       # Signal: Spoofed executive domain targeting Engineering
        'hr@company.com',                # Signal: Legitimate internal HR email with handbook
        'it-service@microsoft-update.net',# Signal: Weaponized executable (.scr) targeting low-scoring user
        'marketing@salesforce.com'       # Signal: Legitimate marketing traffic with large attachment
    ],
    'recipient_id': ['u99', 'u12', 'u12', 'u45', 'u99', 'u12', 'u88', 'u45', 'u45', 'u12', 'u88', 'u45'],
    'email_subject': [
        'Urgent: Update your 401k', 
        'Account Suspended Alert', 
        'Quick task for you', 
        'Daily Tech News', 
        'ACTION REQUIRED: Reset Password', 
        'Your monthly statement',
        'Overdue Invoice Notification', 
        'Immediate Action Required',
        'Are you at your desk?', 
        'New employee handbook policy', 
        'Critical Patch Installation', 
        'Quarterly Leads Report'
    ],
    'timestamp': [
        '2026-05-16 10:00:00', '2026-05-16 10:15:00', '2026-05-16 11:00:00', '2026-05-16 11:30:00',
        '2026-05-16 11:32:00', '2026-05-16 12:05:00', '2026-05-16 12:06:00', '2026-05-16 12:06:30',
        '2026-05-16 14:22:00', '2026-05-16 15:00:00', '2026-05-16 15:05:00', '2026-05-16 16:45:00'
    ],
    'has_attachment': [0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1],
    'attachment_extension': [None, None, None, None, None, '.pdf', '.pdf', '.pdf', None, '.docx', '.scr', '.xlsx'],
    'num_links_in_body': [2, 1, 0, 4, 1, 1, 1, 2, 0, 0, 1, 5]
}
pd.DataFrame(emails_data).to_parquet('inbound_emails.parquet')

# 2. EXPANDED DOMAIN REGISTRY (Global lookup table)
registry_data = {
    'domain_name': [
        'company.com', 'techcrunch.com', 'salesforce.com', 'company-benefits.com', 
        'paypal-verify.net', 'micros0ft.com', 'aws-invoice.io', 'microsoft-update.net'
    ],
    'registration_date': [
        '2010-01-01', '2005-06-12', '1999-03-05', '2026-05-14', # company-benefits is 2 days old
        '2026-05-15', '2026-05-16', '2026-05-15', '2026-05-16'  # newly minted domains
    ],
    'is_verified_brand': [1, 1, 1, 0, 0, 0, 0, 0],
    'domain_country': ['US', 'US', 'US', 'ZA', 'RU', 'CN', 'NL', 'NL']
}
pd.DataFrame(registry_data).to_parquet('domain_registry.parquet')

# 3. USER BEHAVIOR & RISK METRICS (Employee profiles)
user_data = {
    'user_id': ['u12', 'u45', 'u88', 'u99'],
    'department': ['Finance', 'Engineering', 'Customer Success', 'Human Resources'],
    'past_training_score': [85, 98, 45, 92],  # u88 represents a severe risk factor
    'historical_phish_clicks': [1, 0, 4, 0],  # u88 is a high-recidivist clicker
    'is_vip': [0, 0, 0, 1]                     # u99 is the HR Director (high-value target)
}
pd.DataFrame(user_data).to_parquet('user_profiles.parquet')

print("Signal-Rich Parquet files created successfully!")

Signal-Rich Parquet files created successfully!


In [ ]:
# import libraries
import pandas as pd
import numpy as np

# Read parquet files
domain_reg = pd.read_parquet('domain_registry.parquet')
in_emails = pd.read_parquet('inbound_emails.parquet')
user_prof = pd.read_parquet('user_profiles.parquet')

#print('domain_reg shape: ', domain_reg.shape)
#print(domain_reg.info())
#print('in_emails shape: ', in_emails.shape)
#print(in_emails.info())
#print('user_prof shape: ', user_prof.shape)
#print(user_prof.info())

# Domain Match Key: Extract the exact text after the @ in sender_address.
# in_emails['sender_dom'] = in_emails['sender_address'].str.split('@').str[1]
in_emails['domain_name'] = in_emails['sender_address'].str.extract(r'(?:.*)@(.*)')

# Join all dataframes into one df
df = pd.merge(
        (pd.merge(
        in_emails,
        domain_reg,
        on='domain_name',
        how='left'
)), 
    user_prof,
    left_on='recipient_id',
    right_on='user_id',
    how='left'
).drop(columns='user_id')

# Convert relevant columns to datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['registration_date'] = pd.to_datetime(df['registration_date'])

# Convert is_verified_brand to integers - fill NaNs first
df['is_verified_brand'] = df['is_verified_brand'].fillna(99) # Can't use 0 as we don't know
df['is_verified_brand'] = df['is_verified_brand'].astype(int)

# Calculate domain age in days
df['domain_age_days'] = (df['registration_date'] - df['timestamp']).dt.days.abs()

# Create a binary column if an email's sender domain is verified or not
df['is_lookalike_vip_target'] = (
    df['is_verified_brand'].isin([0, 99])
    & (df['is_vip'] == 1)
).astype(int)

# Flag any row where attachment_extension equals .scr or .exe
df['attach_flag'] = (
    df['attachment_extension'].isin(['.scr', '.exe'])
).astype(int)

# For each sender domain, compute the number of emails hitting the gateway within a rolling 5-minute window.
df = df.sort_values(['domain_name', 'timestamp'])
df = df.set_index('timestamp')

# Group by domain and create rolling window
df['burst_count_5m'] = (
    df.groupby('domain_name')['email_id']
      .rolling('5min')
      .count()
      .reset_index(level=0, drop=True)
).astype(int)
df

# Phase 2
# 1. aws-invoice.io
# 2. The sender domain is suspicious, being from microsoft-update.net in combination with a past_training_score of 45 indicates a vulnerability
# 3. It shows a missing value, NaN; all NaN's for the registration date should be filled with something like '1900-01-01 00:00:00' to indicate an abnormality

# Phase 3
# 1. We know that the is_vip for this email is set to 0, meaning that this was no boss from the company and combined with our is_verified_brand set to 99 (missing value) this should be flagged
# 2. Low latency for this will be key to stop attacks as quickly as possible. Pre-aggregating historical risk metrics would be the best solution


,email_id,sender_address,recipient_id,email_subject,has_attachment,attachment_extension,num_links_in_body,domain_name,registration_date,is_verified_brand,domain_country,department,past_training_score,historical_phish_clicks,is_vip,domain_age_days,is_lookalike_vip_target,attach_flag,burst_count_5m
timestamp,,,,,,,,,,,,,,,,,,,
2026-05-16 12:05:00,e6,billing@aws-invoice.io,u12,Your monthly statement,1,.pdf,1,aws-invoice.io,2026-05-15,0,NL,Finance,85,1,0,2.0,0,0,1
2026-05-16 12:06:00,e7,accounting@aws-invoice.io,u88,Overdue Invoice Notification,1,.pdf,1,aws-invoice.io,2026-05-15,0,NL,Customer Success,45,4,0,2.0,0,0,2
2026-05-16 12:06:30,e8,support@aws-invoice.io,u45,Immediate Action Required,1,.pdf,2,aws-invoice.io,2026-05-15,0,NL,Engineering,98,0,0,2.0,0,0,3
2026-05-16 10:00:00,e1,hr@company-benefits.com,u99,Urgent: Update your 401k,0,None,2,company-benefits.com,2026-05-14,0,ZA,Human Resources,92,0,1,3.0,1,0,1
2026-05-16 14:22:00,e9,boss@company-urgent.com,u45,Are you at your desk?,0,None,0,company-urgent.com,NaT,99,NaN,Engineering,98,0,0,NaN,0,0,1
2026-05-16 11:00:00,e3,boss@company.com,u12,Quick task for you,0,None,0,company.com,2010-01-01,1,US,Finance,85,1,0,5980.0,0,0,1
2026-05-16 15:00:00,e10,hr@company.com,u12,New employee handbook policy,1,.docx,0,company.com,2010-01-01,1,US,Finance,85,1,0,5980.0,0,0,1
2026-05-16 11:32:00,e5,admin@micros0ft.com,u99,ACTION REQUIRED: Reset Password,0,None,1,micros0ft.com,2026-05-16,0,CN,Human Resources,92,0,1,1.0,1,0,1
2026-05-16 15:05:00,e11,it-service@microsoft-update.net,u88,Critical Patch Installation,1,.scr,1,microsoft-update.net,2026-05-16,0,NL,Customer Success,45,4,0,1.0,0,1,1
